In [0]:
%python
# Definir pastas do projeto em variáveis para facilitar
bronze_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/'
silver_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/'
gold_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/'
resource_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem'
resource_path_volume = '/Volumes/bikestore/logistics/bikestore_resource/origem/'

In [0]:
%python
# Criando várias tabelas temporárias 
bronze_map = {
    'tmp_bronze_brands': f'{bronze_path}/brands/',
    'tmp_bronze_categories': f'{bronze_path}/categories/',
    'tmp_bronze_customers': f'{bronze_path}/customers/',
    'tmp_bronze_order_items': f'{bronze_path}/order_items/',
    'tmp_bronze_orders': f'{bronze_path}/orders/',
    'tmp_bronze_products': f'{bronze_path}/products/',
    'tmp_bronze_staffs': f'{bronze_path}/staffs/',
    'tmp_bronze_stocks': f'{bronze_path}/stocks/',
    'tmp_bronze_stores': f'{bronze_path}/stores/'
}
for view_name, path in bronze_map.items():
    (spark.read.format('delta')
    .load(path)
    .createOrReplaceTempView(view_name)
) 

In [0]:
%sql
SELECT * FROM tmp_bronze_orders LIMIT 10

order_id,customer_id,order_status,order_date,required_date,shipped_date,store_id,staff_id
1,259,4,2016-01-01,2016-01-03,2016-01-03,1,2
2,1212,4,2016-01-01,2016-01-04,2016-01-03,2,6
3,523,4,2016-01-02,2016-01-05,2016-01-03,2,7
4,175,4,2016-01-03,2016-01-04,2016-01-05,1,3
5,1324,4,2016-01-03,2016-01-06,2016-01-06,2,6
6,94,4,2016-01-04,2016-01-07,2016-01-05,2,6
7,324,4,2016-01-04,2016-01-07,2016-01-05,2,6
8,1204,4,2016-01-04,2016-01-05,2016-01-05,2,7
9,60,4,2016-01-05,2016-01-08,2016-01-08,1,2
10,442,4,2016-01-05,2016-01-06,2016-01-06,2,6


In [0]:
DESCRIBE tmp_bronze_orders

col_name,data_type,comment
order_id,int,null
customer_id,int,null
order_status,int,null
order_date,date,null
required_date,date,null
shipped_date,string,null
store_id,int,null
staff_id,int,null


In [0]:
SELECT * FROM tmp_bronze_stores

store_id,store_name,phone,email,street,city,state,zip_code
1,Santa Cruz Bikes,(831) 476-4321,santacruz@bikes.shop,3700 Portola Drive,Santa Cruz,CA,95060
2,Baldwin Bikes,(516) 379-8888,baldwin@bikes.shop,4200 Chestnut Lane,Baldwin,NY,11432
3,Rowlett Bikes,(972) 530-5555,rowlett@bikes.shop,8000 Fairway Avenue,Rowlett,TX,75088


In [0]:
DESCRIBE tmp_bronze_stores

col_name,data_type,comment
store_id,int,null
store_name,string,null
phone,string,null
email,string,null
street,string,null
city,string,null
state,string,null
zip_code,int,null


In [0]:
SELECT * FROM tmp_bronze_staffs LIMIT 10

staff_id,first_name,last_name,email,phone,active,store_id,manager_id
1,Fabiola,Jackson,fabiola.jackson@bikes.shop,(831) 555-5554,1,1,NULL
2,Mireya,Copeland,mireya.copeland@bikes.shop,(831) 555-5555,1,1,1
3,Genna,Serrano,genna.serrano@bikes.shop,(831) 555-5556,1,1,2
4,Virgie,Wiggins,virgie.wiggins@bikes.shop,(831) 555-5557,1,1,2
5,Jannette,David,jannette.david@bikes.shop,(516) 379-4444,1,2,1
6,Marcelene,Boyer,marcelene.boyer@bikes.shop,(516) 379-4445,1,2,5
7,Venita,Daniel,venita.daniel@bikes.shop,(516) 379-4446,1,2,5
8,Kali,Vargas,kali.vargas@bikes.shop,(972) 530-5555,1,3,1
9,Layla,Terrell,layla.terrell@bikes.shop,(972) 530-5556,1,3,7
10,Bernardine,Houston,bernardine.houston@bikes.shop,(972) 530-5557,1,3,7


In [0]:
DESCRIBE tmp_bronze_staffs

col_name,data_type,comment
staff_id,int,null
first_name,string,null
last_name,string,null
email,string,null
phone,string,null
active,int,null
store_id,int,null
manager_id,string,null


Mapeamento Status <br>
1 THEN 'Pending' - Pendente <br>
2 THEN 'Processing' - Processando <br> 
3 THEN 'Shipped' - Expedido <br>
4 THEN 'Delivered' - Entregue <br>
ELSE 'Unknown' - Desconhecido

In [0]:
SELECT * FROM tmp_bronze_order_items limit 15

order_id,item_id,product_id,quantity,list_price,discount
1,1,20,1,599.99,0.2
1,2,8,2,1799.99,0.07
1,3,10,2,1549.0,0.05
1,4,16,2,599.99,0.05
1,5,4,1,2899.99,0.2
2,1,20,1,599.99,0.07
2,2,16,2,599.99,0.05
3,1,3,1,999.99,0.05
3,2,20,1,599.99,0.05
4,1,2,2,749.99,0.1


In [0]:
DESCRIBE tmp_bronze_order_items 

col_name,data_type,comment
order_id,int,null
item_id,int,null
product_id,int,null
quantity,int,null
list_price,double,null
discount,double,null


In [0]:
SELECT
    OI.order_id,
    OI.item_id,
    OI.product_id,
    OI.quantity,
    OI.list_price ,
    -- (OI.list_price * OI.quantity) AS value_test,
    ROUND((OI.list_price * OI.quantity) * (1-OI.discount), 2) AS total_sale,
    OI.discount
FROM tmp_bronze_order_items AS OI

/*
Explicação:
1 - discount -> Pega 80% do preço original, para 20% de desconto.
list_price * (1 - discount) -> calcula o valor final que o cliente paga.
No exemplo, 599.99 * (1 - 0.2) = 479.992, ou seja, aproximadamente 480.00.
A conta seria 599.99 * (0.8).
*/

order_id,item_id,product_id,quantity,list_price,total_sale,discount
1,1,20,1,599.99,479.99,0.2
1,2,8,2,1799.99,3347.98,0.07
1,3,10,2,1549.0,2943.1,0.05
1,4,16,2,599.99,1139.98,0.05
1,5,4,1,2899.99,2319.99,0.2
2,1,20,1,599.99,557.99,0.07
2,2,16,2,599.99,1139.98,0.05
3,1,3,1,999.99,949.99,0.05
3,2,20,1,599.99,569.99,0.05
4,1,2,2,749.99,1349.98,0.1


In [0]:
DESCRIBE tmp_bronze_order_items 

col_name,data_type,comment
order_id,int,null
item_id,int,null
product_id,int,null
quantity,int,null
list_price,double,null
discount,double,null


In [0]:
SELECT * FROM bikestore.logistics.silver_product
WHERE product_id = 8

product_id,product_name,brand_id,category_name,model_year,list_price,total_stock
8,Trek Remedy 29 Carbon Frameset - 2016,9,Mountain Bikes,2016,1799.99,13


In [0]:
-- Realizando o CTE
WITH order_items as (
    SELECT
    OI.order_id,
    OI.item_id,
    OI.product_id,
    OI.quantity,
    OI.list_price ,
    -- (OI.list_price * OI.quantity) AS value_test,
    ROUND((OI.list_price * OI.quantity) * (1-OI.discount), 2)AS total_sale,
    OI.discount
FROM tmp_bronze_order_items AS OI
)

-- Select final
SELECT
    ORD.order_id,
    ORD.customer_id,
    CASE 
        WHEN ORD.order_status = 1 THEN 'Pending'
        WHEN ORD.order_status = 2 THEN 'Processing'
        WHEN ORD.order_status = 3 THEN 'Shipped'
        WHEN ORD.order_status = 4 THEN 'Delivered'
        ELSE 'Unknown'
    END AS status,
    ORD.order_status,
    ORD.order_date,
    ORD.required_date,
    ORD.shipped_date,
    -- ORD.store_id,
    -- ST.store_id AS store_id_loja,
    ST.store_name,
    ST.state,
    ST.city,
    -- ORD.staff_id,
    -- STF.staff_id AS staff_id_vendedor,
    STF.first_name AS staff_nome,
    STF.active AS active_staff,
    STF.email,
    OIT.product_id,
    OIT.quantity,
    OIT.total_sale,
    OIT.list_price,
    OIT.discount
FROM tmp_bronze_orders AS ORD
LEFT JOIN    tmp_bronze_stores AS ST ON ORD.store_id = ST.store_id
LEFT JOIN   tmp_bronze_staffs AS STF ON ORD.staff_id = STF.staff_id
LEFT JOIN         order_items AS OIT ON ORD.order_id = OIT.order_id



order_id,customer_id,status,order_status,order_date,required_date,shipped_date,store_name,state,city,staff_nome,active_staff,email,product_id,quantity,total_sale,list_price,discount
1,259,Delivered,4,2016-01-01,2016-01-03,2016-01-03,Santa Cruz Bikes,CA,Santa Cruz,Mireya,1,mireya.copeland@bikes.shop,4,1,2319.99,2899.99,0.2
2,1212,Delivered,4,2016-01-01,2016-01-04,2016-01-03,Baldwin Bikes,NY,Baldwin,Marcelene,1,marcelene.boyer@bikes.shop,16,2,1139.98,599.99,0.05
3,523,Delivered,4,2016-01-02,2016-01-05,2016-01-03,Baldwin Bikes,NY,Baldwin,Venita,1,venita.daniel@bikes.shop,20,1,569.99,599.99,0.05
4,175,Delivered,4,2016-01-03,2016-01-04,2016-01-05,Santa Cruz Bikes,CA,Santa Cruz,Genna,1,genna.serrano@bikes.shop,2,2,1349.98,749.99,0.1
5,1324,Delivered,4,2016-01-03,2016-01-06,2016-01-06,Baldwin Bikes,NY,Baldwin,Marcelene,1,marcelene.boyer@bikes.shop,26,1,557.99,599.99,0.07
6,94,Delivered,4,2016-01-04,2016-01-07,2016-01-05,Baldwin Bikes,NY,Baldwin,Marcelene,1,marcelene.boyer@bikes.shop,9,2,5579.98,2999.99,0.07
7,324,Delivered,4,2016-01-04,2016-01-07,2016-01-05,Baldwin Bikes,NY,Baldwin,Marcelene,1,marcelene.boyer@bikes.shop,17,2,772.2,429.0,0.1
8,1204,Delivered,4,2016-01-04,2016-01-05,2016-01-05,Baldwin Bikes,NY,Baldwin,Venita,1,venita.daniel@bikes.shop,20,2,1115.98,599.99,0.07
9,60,Delivered,4,2016-01-05,2016-01-08,2016-01-08,Santa Cruz Bikes,CA,Santa Cruz,Mireya,1,mireya.copeland@bikes.shop,7,2,7199.98,3999.99,0.1
10,442,Delivered,4,2016-01-05,2016-01-06,2016-01-06,Baldwin Bikes,NY,Baldwin,Marcelene,1,marcelene.boyer@bikes.shop,14,1,242.99,269.99,0.1


In [0]:
%python
# Salvando no formato delta na camada Silver
df_orders_silver = spark.sql("""
-- Realizando o CTE
WITH order_items as (
    SELECT
    OI.order_id,
    OI.item_id,
    OI.product_id,
    OI.quantity,
    OI.list_price ,
    -- (OI.list_price * OI.quantity) AS value_test,
    ROUND((OI.list_price * OI.quantity) * (1-OI.discount), 2)AS total_sale,
    OI.discount
FROM tmp_bronze_order_items AS OI
)

-- Select final
SELECT
    ORD.order_id,
    ORD.customer_id,
    CASE 
        WHEN ORD.order_status = 1 THEN 'Pending'
        WHEN ORD.order_status = 2 THEN 'Processing'
        WHEN ORD.order_status = 3 THEN 'Shipped'
        WHEN ORD.order_status = 4 THEN 'Delivered'
        ELSE 'Unknown'
    END AS status,
    ORD.order_status,
    ORD.order_date,
    ORD.required_date,
    ORD.shipped_date,
    -- ORD.store_id,
    -- ST.store_id AS store_id_loja,
    ST.store_name,
    ST.state,
    ST.city,
    -- ORD.staff_id,
    -- STF.staff_id AS staff_id_vendedor,
    STF.first_name AS staff_nome,
    STF.active AS active_staff,
    STF.email,
    OIT.product_id,
    OIT.quantity,
    OIT.total_sale,
    OIT.list_price,
    OIT.discount
FROM tmp_bronze_orders AS ORD
LEFT JOIN    tmp_bronze_stores AS ST ON ORD.store_id = ST.store_id
LEFT JOIN   tmp_bronze_staffs AS STF ON ORD.staff_id = STF.staff_id
LEFT JOIN         order_items AS OIT ON ORD.order_id = OIT.order_id
""")

# Salvar e transformar em formato delta na camada Silver
df_orders_silver.write \
    .mode('overwrite') \
    .format('delta') \
    .option('mergeSchema', 'true') \
    .save(f'{silver_path}/orders')


In [0]:
/*Criar tabela física de produção, este comando precisa rodar só 1 vez, pois quando os arquivos forem atualizados a tabela refletirá o resultado (Vai obter o log mais recente e trazer dados do(s) arquivo(s) atualizado)*/
CREATE TABLE IF NOT EXISTS bikestore.logistics.silver_orders
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/orders';

In [0]:
SELECT * FROM bikestore.logistics.silver_product LIMIT 15

product_id,product_name,brand_id,category_name,model_year,list_price,total_stock
1,Trek 820 - 2016,9,Mountain Bikes,2016,379.99,55
2,Ritchey Timberwolf Frameset - 2016,5,Mountain Bikes,2016,749.99,45
3,Surly Wednesday Frameset - 2016,8,Mountain Bikes,2016,999.99,34
4,Trek Fuel EX 8 29 - 2016,9,Mountain Bikes,2016,2899.99,36
5,Heller Shagamaw Frame - 2016,3,Mountain Bikes,2016,1320.99,26
6,Surly Ice Cream Truck Frameset - 2016,8,Mountain Bikes,2016,469.99,38
7,Trek Slash 8 27.5 - 2016,9,Mountain Bikes,2016,3999.99,28
8,Trek Remedy 29 Carbon Frameset - 2016,9,Mountain Bikes,2016,1799.99,13
9,Trek Conduit+ - 2016,9,Electric Bikes,2016,2999.99,51
10,Surly Straggler - 2016,8,Cyclocross Bicycles,2016,1549.0,49
